In [ ]:
import pandas as pd
from Bio import SeqIO
from matplotlib import pyplot as plt
import statistics

#read in a gff annotation as a dataframe
def read_gff(file):
    return pd.read_csv(file, sep="\t", comment="#", names=["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"])

annotation = read_gff('PO2744_Nanomia_bijuga.annotation.gff')
annotation = annotation.sort_values(by=['seqid', 'start']).reset_index(drop=True)

#retain only the rows in which 'type'=='gene'
genes = annotation[annotation['type']=='gene']
genes = genes.reset_index(drop=True)

#subset the dataframe to only contain seqid, start, end, and attributes columns
genes = genes[['seqid', 'start', 'end', 'attributes']]
#check the dimensions of genes to know the number of genes in the annotation with scaffolds
print(genes.shape)
#remove the seqids that end with scaf*
genes = genes[~genes['seqid'].str.contains('scaf')]
genes = genes.reset_index(drop=True)
print(genes)
print(genes.shape)

In [ ]:
#subset each unique seqid
seqids = genes['seqid'].unique()
#make a list of different windows to use to calculate gene density
windows = [1000000, 5000000, 10000000, 25000000]
#check the chromosome sizes
#open the genome file in fasta format and print the length of the seqid
genome = SeqIO.parse('NS_hap2_v0.8.fasta', 'fasta')
#get the length of each seqid and make a dictionary for each id and sequence length
seq_length = {}
for seq_record in genome:
    if seq_record.id in seqids:
        print(seq_record.id)
        print(len(seq_record.seq))
        #make a dictionary of id and sequence length
        seq_length[seq_record.id] = len(seq_record.seq)
print(seq_length)



In [ ]:
#make a for loop that goes through all the rows with the same seqid
for seqid in seqids:
    gene_density = pd.DataFrame(columns=['seqid', 'window', 'start', 'end', 'gene_number', 'density'])
    #test_info = pd.DataFrame(columns=['prev_end', 'start', 'distance_between_genes'])
    seqid_genes = genes[genes['seqid']==seqid]
    seqid_genes = seqid_genes.reset_index(drop=True)
    num_of_genes_chr = seqid_genes.shape[0]
    print(seqid, num_of_genes_chr)
    #Calculate chromosome length here
    chr_length = seq_length[seqid]
    genes_in_window = []
    for window in windows:
        gene_distance = []
        range_start=0
        range_end=window
        for i in range(len(seqid_genes)):
            start = seqid_genes.loc[i, 'start']
            end = seqid_genes.loc[i, 'end']
            if i>0:
                prev_end = seqid_genes.loc[i-1, 'end']
                distance_between_genes = start - prev_end
                if distance_between_genes<0:
                    distance_between_genes = 0
                gene_distance.append(distance_between_genes)
                #test_info.loc[len(test_info)] = [prev_end, start, distance_between_genes]
            if start>range_start and end<range_end:
                genes_in_window.append(seqid_genes.loc[i, 'attributes'])
            else:
                gene_num = len(genes_in_window)
                ratio = gene_num/window
                if range_start==0:
                    gene_density.loc[len(gene_density)] = [seqid, window, range_start, range_end, gene_num, ratio]
                    range_start = window + 1
                    range_end = range_end + window
                else:
                    gene_density.loc[len(gene_density)] = [seqid, window, range_start, range_end, gene_num, ratio]
                    range_start = range_start + window
                    range_end = range_end + window
                if range_end>chr_length:
                    range_end = chr_length
                genes_in_window = []

        #create a plot where x are the chromosomal coordinates and y are the gene densities
        to_plot = gene_density[gene_density['window']==window]
        to_plot = to_plot.reset_index(drop=True)
        plt.figure()
        plt.plot(to_plot['start'], to_plot['gene_number'])
        plt.title(seqid+' ' + str(window) + 'bp window')
        plt.xlabel('Chromosomal coordinates')
        plt.ylabel('Gene number')
    plt.figure()
    median_dist_genes = statistics.median(gene_distance)
    plt.hist(gene_distance, bins=100, range = (0, 500000))
    plt.axvline(median_dist_genes, color='red', linestyle='dashed', linewidth=1)
    plt.title(seqid+' Histogram of gene distances')
    print(gene_density)